In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'

import numpy as np
import jax
import jax.numpy as jnp
import gc

jax.config.update('jax_enable_x64', True)

from bayesianquilts.data.lung import get_data
from bayesianquilts.jax.parameter import Interactions
from bayesianquilts.predictors.survival import PiecewiseExponentialQuilt, PiecewiseExponentialLikelihood
from bayesianquilts.metrics.ais import AdaptiveImportanceSampler

## Load lung cancer data

In [ ]:
df, n_obs = get_data(cache_dir='/tmp/lung_cache')
print(f'Observations: {n_obs}')
print(f'Events: {df["event"].sum()}')
print(f'Censored: {n_obs - df["event"].sum()}')
print(df.head(10))

## Define model

In [ ]:
# Breakpoints at quantiles of uncensored event times
event_times = df.filter(df["event"] == 1)["time"].to_numpy()
q25, q50, q75 = np.quantile(event_times, [0.25, 0.50, 0.75])
breakpoints = np.array([q25, q50, q75])
print(f'Breakpoints (25/50/75 pctile of event times): {breakpoints}')
print(f'Intervals: [0, {breakpoints[0]}], ({breakpoints[0]}, {breakpoints[1]}], ({breakpoints[1]}, {breakpoints[2]}], ({breakpoints[2]}, inf)')

# Interactions: sex (2) x age_group (3) x ecog_group (2) = 12 groups
rate_interact = Interactions(dimensions=[
    ("sex_idx", 2),
    ("age_group", 3),
    ("ecog_group", 2),
])

model = PiecewiseExponentialQuilt(
    breakpoints=breakpoints,
    rate_interact=rate_interact,
    shrinkage_scale=4e-2,
    dim_decay_factor=0.9,
    dtype=jnp.float64,
)

print(f'\nRate decomposition:\n{model.rate_decomposition}')
print(f'Surrogate params: {len(model.params)} keys')
print(f'Var list: {model.var_list[:5]}...')

## Prepare data and fit model

In [ ]:
# Prepare data dict — each column as (N, 1) int array for Decomposed compatibility
data = {}
for col in ["sex_idx", "age_group", "ecog_group"]:
    data[col] = df[col].to_numpy()[:, None]

data["time"] = df["time"].to_numpy().astype(np.float64)
data["event"] = df["event"].to_numpy().astype(np.float64)

print(f'Data shapes:')
for k, v in data.items():
    print(f'  {k}: {v.shape} {v.dtype}')

In [ ]:
# Full-batch data factory (small dataset, no need for minibatching)
batch_size = n_obs

def data_factory():
    yield data

losses, _ = model.fit(
    data_factory,
    batch_size=batch_size,
    dataset_size=n_obs,
    num_epochs=200,
    learning_rate=5e-3,
    lr_decay_factor=0.975,
    patience=10,
    warmup_max_order=3,
    epochs_per_stage=25,
)
print(f'Final loss: {losses[-1]:.2f}')

## Sample posterior and compute baseline PSIS khat

In [ ]:
n_samples = 1000
surrogate = model.surrogate_distribution_generator(model.params)
key = jax.random.PRNGKey(42)
samples = surrogate.sample(n_samples, seed=key)

print(f'Posterior sample keys: {list(samples.keys())[:5]}...')
print(f'Sample shapes:')
for k, v in list(samples.items())[:5]:
    print(f'  {k}: {v.shape}')

In [ ]:
# Baseline: identity PSIS (no transformation)
likelihood_fn = PiecewiseExponentialLikelihood(
    breakpoints=breakpoints,
    rate_decomposition=model.rate_decomposition,
    dtype=jnp.float64,
)

ais_sampler = AdaptiveImportanceSampler(likelihood_fn=likelihood_fn)

rhos = [2**-r for r in range(-2, 12)]

results_identity = ais_sampler.adaptive_is_loo(
    data=data,
    params=samples,
    rhos=rhos,
    variational=False,
    transformations=['identity'],
)

khat_identity = np.array(results_identity['identity']['khat'])
n_bad = int(np.sum(khat_identity >= 0.7))
print(f'Identity PSIS: {n_bad}/{n_obs} observations with khat >= 0.7')
print(f'khat quantiles: {np.quantile(khat_identity, [0.25, 0.5, 0.75, 0.9, 0.95, 1.0])}')

## AIS LOO-CV

In [ ]:
import pandas as pd

n_simulations = 1
base_transform = ['identity']
other_transforms = ['ll', 'kl', 'var', 'mm1', 'mm2', 'pmm1', 'pmm2']

simulation_records = []

for i in range(n_simulations):
    key = jax.random.PRNGKey(i * 1000)
    params = surrogate.sample(n_samples, seed=key)

    # Identity baseline
    results_base = ais_sampler.adaptive_is_loo(
        data=data, params=params,
        rhos=rhos, variational=False,
        transformations=base_transform,
    )
    khat_id = np.array(results_base['identity']['khat'])
    idx_bad = np.where(khat_id >= 0.7)[0]
    print(f'Sim {i+1}: {len(idx_bad)}/{n_obs} observations need adaptation')

    del results_base; gc.collect()

    method_khats = {'identity': khat_id}
    for m in other_transforms:
        method_khats[m] = khat_id.copy()

    if len(idx_bad) > 0:
        # Run AIS transformations on problematic observations
        # Subset data to bad observations
        mask = np.zeros(n_obs, dtype=bool)
        mask[idx_bad] = True

        data_sub = {
            'sex_idx': data['sex_idx'][mask],
            'age_group': data['age_group'][mask],
            'ecog_group': data['ecog_group'][mask],
            'time': data['time'][mask],
            'event': data['event'][mask],
        }

        results_sub = ais_sampler.adaptive_is_loo(
            data=data_sub, params=params,
            rhos=rhos, variational=False,
            transformations=other_transforms,
        )

        for base_method in other_transforms:
            khat_arrays = []
            for key_name, res in results_sub.items():
                if key_name == 'best':
                    continue
                if key_name == base_method or key_name.startswith(base_method + '_'):
                    khat_arrays.append(res['khat'])
            if khat_arrays:
                min_khat = np.array(np.min(np.stack(khat_arrays), axis=0))
                method_khats[base_method][idx_bad] = min_khat

        del results_sub; gc.collect()

    # Summarize
    groups = {
        'Base': ['identity'],
        'PMM1': ['pmm1'], 'PMM2': ['pmm2'],
        'KL': ['kl'], 'Var': ['var'], 'LL': ['ll'],
        'Ours_Combined': ['pmm1', 'pmm2', 'kl', 'var', 'll'],
        'MM1': ['mm1'], 'MM2': ['mm2'],
        'Full': other_transforms + ['identity'],
    }

    sim_counts = {}
    for gname, methods in groups.items():
        grouped = [method_khats[m] for m in methods if m in method_khats]
        if grouped:
            best = np.min(np.stack(grouped), axis=0)
            sim_counts[gname] = int(np.sum(best > 0.7))
        else:
            sim_counts[gname] = np.nan
    simulation_records.append(sim_counts)

df_results = pd.DataFrame(simulation_records)
print(f'\n--- Piecewise Exponential Survival: Unsuccessful Adaptations ---')
print(df_results)

## Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# khat histogram
axes[0].hist(khat_identity, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(0.7, color='red', linestyle='--', label='khat=0.7')
axes[0].set_xlabel('khat')
axes[0].set_ylabel('Count')
axes[0].set_title('Identity PSIS khat distribution')
axes[0].legend()

# khat vs observation index
axes[1].scatter(range(n_obs), khat_identity, s=10, alpha=0.7)
axes[1].axhline(0.7, color='red', linestyle='--', label='khat=0.7')
axes[1].set_xlabel('Observation index')
axes[1].set_ylabel('khat')
axes[1].set_title('khat per observation')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  Total observations: {n_obs}')
print(f'  khat >= 0.7 (identity): {n_bad}')
print(f'  khat >= 1.0 (identity): {int(np.sum(khat_identity >= 1.0))}')
print(f'  Mean khat: {np.mean(khat_identity):.3f}')
print(f'  Max khat: {np.max(khat_identity):.3f}')